In [1]:
#2021.10.20. TUE
#Team_SmokeFree

## SCORE CALCULATION FOR MCLP
#00. 패키지 호출하기. 
import warnings
import re
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import MinMaxScaler

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')


In [2]:
#01. 데이터셋 전처리하기. 
#(1) RANDOM_POINT 데이터셋 불러오기. 
RANDOM_POINT = pd.read_excel('../data/RANDOM_POINT_DATASET_pp.xlsx')

#(2) 세대 컬럼 스케일링 처리하기. 
scaler = MinMaxScaler()
세대_scaler = scaler.fit_transform(RANDOM_POINT[['세대_COUNT']])

#(3) 값 컬럼 변경하기. 
RANDOM_POINT['세대_COUNT'] = 세대_scaler

#(4) 처리 결과 확인하기. 
RANDOM_POINT

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,음식점_외식_COUNT,음식점_회식_COUNT,음식점_카페_COUNT,위도,경도,geometry
0,0,금천구,1154568000,시흥제2동,0.250891,1,0,0,2,1,9,3,1,37.449962,126.913406,POINT (192337.7432523154 438954.0766042818)
1,1,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449964,126.914394,POINT (192425.2336731556 438954.2123132667)
2,2,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449939,126.914258,POINT (192413.1979568179 438951.4329210447)
3,3,금천구,1154568000,시흥제2동,0.250891,2,0,0,2,1,9,4,1,37.449956,126.913289,POINT (192327.4056526743 438953.4538745985)
4,4,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,12,4,1,37.450008,126.913638,POINT (192358.2717906134 438959.210054699)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40286,41191,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.499947,126.953611,POINT (195897.9891959261 444499.2420208962)
40287,41192,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500272,126.953683,POINT (195904.4341447904 444535.2696354929)
40288,41193,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500105,126.953733,POINT (195908.8060317755 444516.8050831873)
40289,41194,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,1,0,0,37.500540,126.953708,POINT (195906.6493213855 444565.0878704682)


In [3]:
#(5) SHAP Value 데이터셋 불러오기.
SHAP_VAL = pd.read_excel('../data/SHAP_values_dong.xlsx')

#(6) 처리 결과 확인하기.
SHAP_VAL

,행정동_코드,행정동,세대,노래연습장,당구장,PC방,금연구역_10m,금연구역_50m,음식점_외식,음식점_회식,음식점_카페
0,1171063100,가락1동,2.597895,-0.543748,3.325141,7.194775,-0.786023,2.121663,-21.166832,3.484164,0.289485
1,1171063200,가락2동,19.447301,3.531995,-2.902738,-2.953587,-1.925279,1.004377,1.017861,4.509757,0.846309
2,1171062000,가락본동,-5.109760,11.354683,3.083922,-1.364136,5.060548,-1.755528,41.061920,0.603275,-0.008842
3,1153059500,가리봉동,13.575740,8.590204,-5.402833,-2.642563,7.951847,-0.325688,48.876958,-6.381114,0.603056
4,1154551000,가산동,-27.700646,1.911541,1.465872,-0.153733,0.699950,1.260090,23.791659,0.545698,0.412044
...,...,...,...,...,...,...,...,...,...,...,...
419,1117058000,효창동,-11.647341,-1.532828,-1.033294,4.711256,-0.620750,-2.006127,-26.066835,1.748712,1.855414
420,1117051000,후암동,27.822540,-3.551473,-5.373569,0.209138,-4.905190,0.497515,21.998067,2.671751,-2.144422
421,1123072000,휘경제1동,5.479156,0.914973,0.042831,0.825690,3.453579,0.548212,-23.843397,1.595402,0.854290
422,1123073000,휘경제2동,45.237879,-3.438942,-5.453401,0.296867,0.190012,-3.168339,-23.628410,3.837766,2.349718


In [4]:
#(7) BETA 계수 계산하기. 
BETA_VAL = pd.DataFrame({
    '세대'         : 254.41,
    '노래연습장'   : 81.15, 
    '당구장'       : 27.72, 
    'PC방'         : -48.01, 
    '금연구역_10m' : 38.89, 
    '금연구역_50m' : -41.05, 
    '음식점_외식'  : 264.28, 
    '음식점_회식'  : -45.1, 
    '음식점_카페'  : 74.67 
}, index=[0])

#(8) 처리 결과 확인하기.
BETA_VAL

,세대,노래연습장,당구장,PC방,금연구역_10m,금연구역_50m,음식점_외식,음식점_회식,음식점_카페
0,254.41,81.15,27.72,-48.01,38.89,-41.05,264.28,-45.1,74.67


In [5]:
#02. SCORE_b 계산하기. 
#(1) 처리하기. 
RANDOM_POINT['SCORE_b'] = 0

for i in range(len(RANDOM_POINT)) :
    세대_val         = RANDOM_POINT['세대_COUNT'][i] * BETA_VAL['세대'][0]
    노래연습장_val   = RANDOM_POINT['노래연습장_COUNT'][i] * BETA_VAL['노래연습장'][0]
    당구장_val       = RANDOM_POINT['당구장_COUNT'][i] * BETA_VAL['당구장'][0]
    PC방_val         = RANDOM_POINT['PC방_COUNT'][i] * BETA_VAL['PC방'][0]
    금연구역_10m_val = RANDOM_POINT['금연구역_10m_COUNT'][i] * BETA_VAL['금연구역_10m'][0]
    금연구역_50m_val = RANDOM_POINT['금연구역_50m_COUNT'][i] * BETA_VAL['금연구역_50m'][0]
    음식점_외식_val  = RANDOM_POINT['음식점_외식_COUNT'][i] * BETA_VAL['음식점_외식'][0]
    음식점_회식_val  = RANDOM_POINT['음식점_회식_COUNT'][i] * BETA_VAL['음식점_회식'][0]
    음식점_카페_val  = RANDOM_POINT['음식점_카페_COUNT'][i] * BETA_VAL['음식점_카페'][0]
    BETA_SUM         = 세대_val + 노래연습장_val + 당구장_val + PC방_val + 금연구역_10m_val + 금연구역_50m_val + 음식점_외식_val + 음식점_회식_val + 음식점_카페_val

    RANDOM_POINT['SCORE_b'][i] = BETA_SUM
    print(f'DONE ! INDEX_NUM = {i}')

DONE ! INDEX_NUM = 0
DONE ! INDEX_NUM = 1
DONE ! INDEX_NUM = 2
DONE ! INDEX_NUM = 3
DONE ! INDEX_NUM = 4
DONE ! INDEX_NUM = 5
DONE ! INDEX_NUM = 6
DONE ! INDEX_NUM = 7
DONE ! INDEX_NUM = 8
DONE ! INDEX_NUM = 9
DONE ! INDEX_NUM = 10
DONE ! INDEX_NUM = 11
DONE ! INDEX_NUM = 12
DONE ! INDEX_NUM = 13
DONE ! INDEX_NUM = 14
DONE ! INDEX_NUM = 15
DONE ! INDEX_NUM = 16
DONE ! INDEX_NUM = 17
DONE ! INDEX_NUM = 18
DONE ! INDEX_NUM = 19
DONE ! INDEX_NUM = 20
DONE ! INDEX_NUM = 21
DONE ! INDEX_NUM = 22
DONE ! INDEX_NUM = 23
DONE ! INDEX_NUM = 24
DONE ! INDEX_NUM = 25
DONE ! INDEX_NUM = 26
DONE ! INDEX_NUM = 27
DONE ! INDEX_NUM = 28
DONE ! INDEX_NUM = 29
DONE ! INDEX_NUM = 30
DONE ! INDEX_NUM = 31
DONE ! INDEX_NUM = 32
DONE ! INDEX_NUM = 33
DONE ! INDEX_NUM = 34
DONE ! INDEX_NUM = 35
DONE ! INDEX_NUM = 36
DONE ! INDEX_NUM = 37
DONE ! INDEX_NUM = 38
DONE ! INDEX_NUM = 39
DONE ! INDEX_NUM = 40
DONE ! INDEX_NUM = 41
DONE ! INDEX_NUM = 42
DONE ! INDEX_NUM = 43
DONE ! INDEX_NUM = 44
DONE ! INDEX_NUM = 4

In [6]:
#(2) 처리 결과 확인하기.
RANDOM_POINT

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,음식점_외식_COUNT,음식점_회식_COUNT,음식점_카페_COUNT,위도,경도,geometry,SCORE_b
0,0,금천구,1154568000,시흥제2동,0.250891,1,0,0,2,1,9,3,1,37.449962,126.913406,POINT (192337.7432523154 438954.0766042818),2499.000000
1,1,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449964,126.914394,POINT (192425.2336731556 438954.2123132667),2981.000000
2,2,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449939,126.914258,POINT (192413.1979568179 438951.4329210447),2981.000000
3,3,금천구,1154568000,시흥제2동,0.250891,2,0,0,2,1,9,4,1,37.449956,126.913289,POINT (192327.4056526743 438953.4538745985),2535.000000
4,4,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,12,4,1,37.450008,126.913638,POINT (192358.2717906134 438959.210054699),3246.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40286,41191,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.499947,126.953611,POINT (195897.9891959261 444499.2420208962),246.316215
40287,41192,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500272,126.953683,POINT (195904.4341447904 444535.2696354929),246.316215
40288,41193,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500105,126.953733,POINT (195908.8060317755 444516.8050831873),246.316215
40289,41194,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,1,0,0,37.500540,126.953708,POINT (195906.6493213855 444565.0878704682),510.596215


In [11]:
#02. SCORE_s 계산하기. 
#(1) 처리하기. 
RANDOM_POINT['SCORE_s'] = 0

for i in range(len(RANDOM_POINT))                                           :
    for k in range(len(SHAP_VAL))                                           :
        if RANDOM_POINT['행정동_코드'][i] == SHAP_VAL['행정동_코드'][k] :    
            세대_val         = RANDOM_POINT['세대_COUNT'][i] * SHAP_VAL['세대'][k]
            노래연습장_val   = RANDOM_POINT['노래연습장_COUNT'][i] * SHAP_VAL['노래연습장'][k]
            당구장_val       = RANDOM_POINT['당구장_COUNT'][i] * SHAP_VAL['당구장'][k]
            PC방_val         = RANDOM_POINT['PC방_COUNT'][i] * SHAP_VAL['PC방'][k]
            금연구역_10m_val = RANDOM_POINT['금연구역_10m_COUNT'][i] * SHAP_VAL['금연구역_10m'][k]
            금연구역_50m_val = RANDOM_POINT['금연구역_50m_COUNT'][i] * SHAP_VAL['금연구역_50m'][k]
            음식점_외식_val  = RANDOM_POINT['음식점_외식_COUNT'][i] * SHAP_VAL['음식점_외식'][k]
            음식점_회식_val  = RANDOM_POINT['음식점_회식_COUNT'][i] * SHAP_VAL['음식점_회식'][k]
            음식점_카페_val  = RANDOM_POINT['음식점_카페_COUNT'][i] * SHAP_VAL['음식점_카페'][k]
            SHAP_SUM         = 세대_val + 노래연습장_val + 당구장_val + PC방_val + 금연구역_10m_val + 금연구역_50m_val + 음식점_외식_val + 음식점_회식_val + 음식점_카페_val

    RANDOM_POINT['SCORE_s'][i] = SHAP_SUM
    print(f'DONE ! INDEX_NUM = {i}')

DONE ! INDEX_NUM = 0
DONE ! INDEX_NUM = 1
DONE ! INDEX_NUM = 2
DONE ! INDEX_NUM = 3
DONE ! INDEX_NUM = 4
DONE ! INDEX_NUM = 5
DONE ! INDEX_NUM = 6
DONE ! INDEX_NUM = 7
DONE ! INDEX_NUM = 8
DONE ! INDEX_NUM = 9
DONE ! INDEX_NUM = 10
DONE ! INDEX_NUM = 11
DONE ! INDEX_NUM = 12
DONE ! INDEX_NUM = 13
DONE ! INDEX_NUM = 14
DONE ! INDEX_NUM = 15
DONE ! INDEX_NUM = 16
DONE ! INDEX_NUM = 17
DONE ! INDEX_NUM = 18
DONE ! INDEX_NUM = 19
DONE ! INDEX_NUM = 20
DONE ! INDEX_NUM = 21
DONE ! INDEX_NUM = 22
DONE ! INDEX_NUM = 23
DONE ! INDEX_NUM = 24
DONE ! INDEX_NUM = 25
DONE ! INDEX_NUM = 26
DONE ! INDEX_NUM = 27
DONE ! INDEX_NUM = 28
DONE ! INDEX_NUM = 29
DONE ! INDEX_NUM = 30
DONE ! INDEX_NUM = 31
DONE ! INDEX_NUM = 32
DONE ! INDEX_NUM = 33
DONE ! INDEX_NUM = 34
DONE ! INDEX_NUM = 35
DONE ! INDEX_NUM = 36
DONE ! INDEX_NUM = 37
DONE ! INDEX_NUM = 38
DONE ! INDEX_NUM = 39
DONE ! INDEX_NUM = 40
DONE ! INDEX_NUM = 41
DONE ! INDEX_NUM = 42
DONE ! INDEX_NUM = 43
DONE ! INDEX_NUM = 44
DONE ! INDEX_NUM = 4

In [12]:
#(2) 처리 결과 확인하기.
RANDOM_POINT

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,음식점_외식_COUNT,음식점_회식_COUNT,음식점_카페_COUNT,위도,경도,geometry,SCORE_b,SCORE_s,SCORE_sum
0,0,금천구,1154568000,시흥제2동,0.250891,1,0,0,2,1,9,3,1,37.449962,126.913406,POINT (192337.7432523154 438954.0766042818),2499.000000,-294,2143.000000
1,1,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449964,126.914394,POINT (192425.2336731556 438954.2123132667),2981.000000,-348,2553.000000
2,2,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449939,126.914258,POINT (192413.1979568179 438951.4329210447),2981.000000,-348,2553.000000
3,3,금천구,1154568000,시흥제2동,0.250891,2,0,0,2,1,9,4,1,37.449956,126.913289,POINT (192327.4056526743 438953.4538745985),2535.000000,-296,2182.000000
4,4,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,12,4,1,37.450008,126.913638,POINT (192358.2717906134 438959.210054699),3246.000000,-380,2777.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40286,41191,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.499947,126.953611,POINT (195897.9891959261 444499.2420208962),246.316215,29,200.913892
40287,41192,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500272,126.953683,POINT (195904.4341447904 444535.2696354929),246.316215,29,200.913892
40288,41193,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500105,126.953733,POINT (195908.8060317755 444516.8050831873),246.316215,29,200.913892
40289,41194,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,1,0,0,37.500540,126.953708,POINT (195906.6493213855 444565.0878704682),510.596215,6,465.193892


In [15]:
#04. SCORE_b와 SCORE_s 합산하기
#(1)
RANDOM_POINT['SCORE_sum'] = (RANDOM_POINT['SCORE_b'] + RANDOM_POINT['SCORE_s'])/2

#(2)
RANDOM_POINT

,rand_point,자치구,행정동_코드,행정동,세대_COUNT,노래연습장_COUNT,당구장_COUNT,PC방_COUNT,금연구역_10m_COUNT,금연구역_50m_COUNT,음식점_외식_COUNT,음식점_회식_COUNT,음식점_카페_COUNT,위도,경도,geometry,SCORE_b,SCORE_s,SCORE_sum
0,0,금천구,1154568000,시흥제2동,0.250891,1,0,0,2,1,9,3,1,37.449962,126.913406,POINT (192337.7432523154 438954.0766042818),2499.000000,-294,1102.500000
1,1,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449964,126.914394,POINT (192425.2336731556 438954.2123132667),2981.000000,-348,1316.500000
2,2,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,11,4,1,37.449939,126.914258,POINT (192413.1979568179 438951.4329210447),2981.000000,-348,1316.500000
3,3,금천구,1154568000,시흥제2동,0.250891,2,0,0,2,1,9,4,1,37.449956,126.913289,POINT (192327.4056526743 438953.4538745985),2535.000000,-296,1119.500000
4,4,금천구,1154568000,시흥제2동,0.250891,0,0,0,3,0,12,4,1,37.450008,126.913638,POINT (192358.2717906134 438959.210054699),3246.000000,-380,1433.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40286,41191,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.499947,126.953611,POINT (195897.9891959261 444499.2420208962),246.316215,29,137.658107
40287,41192,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500272,126.953683,POINT (195904.4341447904 444535.2696354929),246.316215,29,137.658107
40288,41193,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,0,0,0,37.500105,126.953733,POINT (195908.8060317755 444516.8050831873),246.316215,29,137.658107
40289,41194,동작구,1159053000,상도제1동,0.968186,0,0,0,0,0,1,0,0,37.500540,126.953708,POINT (195906.6493213855 444565.0878704682),510.596215,6,258.298107


In [16]:
#(3) 처리 결과 저장하기. 
RANDOM_POINT.to_excel('temp_file.xlsx', index=False)